# Lab 2: The Decision Machine
## When Algorithms Say Yes or No to Your Future

---

### The Story

You're now the **chief data scientist at a bank**. Every day, hundreds of loan applications arrive. Your team needs a system that says **YES** or **NO** — and you need to **JUSTIFY every decision**.

In **Lab 0**, you understood the data. The 1,000 people in the German Credit dataset revealed their financial lives: checking accounts, savings, employment history, age, purpose of the loan.

In **Lab 1**, you predicted how much credit they needed.

Now in **Lab 2**, you DECIDE. Will you approve or reject their loan?

**The challenge:** Every algorithm choice is a policy decision. Every threshold you set determines whose future changes. This is where data science meets ethics, economics, and law.

Let's begin.

### Setup & Data Loading

We're using the German Credit dataset. The target variable is **Risk**: "good" or "bad" credit risk.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
print("All libraries imported. Ready to decide.")

In [ ]:
# Load the German Credit dataset (CSV version with headers)
url = "https://raw.githubusercontent.com/sampathlonka/ml-workshop/master/Data/german_credit_data.csv"
df = pd.read_csv(url)

print(f"Dataset loaded: {df.shape[0]} loan applications, {df.shape[1]} features")
print(f"\nFirst few rows:")
print(df.head())
print(f"\nTarget distribution:")
print(df['Risk'].value_counts())

In [ ]:
# Detect the format of the Risk column and create a binary target
# The Risk column may have values like 1/2, 1/0, or "good"/"bad"

risk_values = df['Risk'].unique()
print(f"Unique Risk values: {risk_values}")
print(f"Risk dtype: {df['Risk'].dtype}")

# Create binary target: 1 = good risk, 0 = bad risk
if set(risk_values).issubset({1, 2}):
    # Format: 1 = good, 2 = bad
    y = (df['Risk'] == 1).astype(int)
    print("Risk format detected: 1/2 (1=good, 2=bad)")
elif set(risk_values).issubset({0, 1}):
    # Format: 1 = good, 0 = bad
    y = df['Risk'].astype(int)
    print("Risk format detected: 0/1 (1=good, 0=bad)")
elif set(str(v).lower() for v in risk_values).issubset({'good', 'bad'}):
    # Format: "good"/"bad"
    y = (df['Risk'].str.lower() == 'good').astype(int)
    print("Risk format detected: 'good'/'bad' strings")
else:
    # Fallback: use LabelEncoder
    le_target = LabelEncoder()
    y = le_target.fit_transform(df['Risk'])
    print(f"Risk format: other format, used LabelEncoder. Classes: {le_target.classes_}")

print(f"\nBinary target created. Good risks (1): {y.sum()}, Bad risks (0): {(1-y).sum()}")

---

# ACT 1: LOGISTIC REGRESSION — The Probability of Trust

## The Frame

Logistic regression doesn't just say **yes** or **no**. It gives you a **probability**: *"This person has a 73% chance of being a good credit risk."*

That probability is power. Because now you can ask: *"Where should I draw the line? Approve everyone above 60%? Or be strict and only approve those above 80%?"*

Every threshold is a **policy decision** with real consequences.

## Prepare the Features

In [ ]:
# Select key features for decision-making (intuitive and policy-relevant)
feature_cols = ['CheckingStatus', 'LoanDuration', 'CreditHistory', 'ExistingSavings', 
                'EmploymentDuration', 'Age', 'LoanAmount', 'LoanPurpose']

X = df[feature_cols].copy()

# Encode categorical variables
le_dict = {}
for col in X.select_dtypes(include='object').columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    le_dict[col] = le

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print(f"Training set: {X_train.shape[0]} applications | Test set: {X_test.shape[0]} applications")
print(f"\nFeatures used for classification:")
for col in feature_cols:
    print(f"  - {col}")

## Train Logistic Regression

This simple yet powerful model will give us **probabilities** for each application.

In [ ]:
# Train logistic regression
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train, y_train)

# Get probability predictions (probability of being a GOOD risk)
y_pred_proba = lr_model.predict_proba(X_test)[:, 1]
y_pred = lr_model.predict(X_test)

print(f"Logistic Regression trained.")
print(f"\nSample predictions (first 5 applicants):")
for i in range(5):
    print(f"  Applicant {i+1}: {y_pred_proba[i]:.1%} chance of being a good risk → Decision: {'APPROVE ✓' if y_pred[i]==1 else 'REJECT ✗'}")

## Visualize: The Distribution of Trust

Where are the decisions? How many people fall into the gray zone?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of probabilities
axes[0].hist(y_pred_proba, bins=30, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(0.5, color='red', linestyle='--', linewidth=2, label='Default threshold (50%)')
axes[0].set_xlabel('Probability of Good Credit Risk', fontsize=11)
axes[0].set_ylabel('Number of Applicants', fontsize=11)
axes[0].set_title('Logistic Regression: Distribution of Trust', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Approved vs Rejected by actual outcome
results_df = pd.DataFrame({
    'Predicted_Risk': y_pred_proba,
    'Actual_Risk': y_test.values,
    'Decision': ['APPROVE' if p > 0.5 else 'REJECT' for p in y_pred_proba]
})

decision_counts = results_df['Decision'].value_counts()
axes[1].bar(decision_counts.index, decision_counts.values, color=['#2ecc71', '#e74c3c'], alpha=0.8, edgecolor='black', linewidth=1.5)
axes[1].set_ylabel('Number of Applicants', fontsize=11)
axes[1].set_title('Loan Decisions at 50% Threshold', fontsize=12, fontweight='bold')
axes[1].grid(alpha=0.3, axis='y')

for i, (label, count) in enumerate(zip(decision_counts.index, decision_counts.values)):
    axes[1].text(i, count + 2, str(count), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## POLICY INSIGHT: The Threshold Decision ⚖️

**Probability = Risk Level.** A bank might approve anyone with probability > 60%. But what if we set it to 80%? Who gets excluded?

Let's compare two thresholds:

In [ ]:
# Compare two threshold strategies
thresholds = [0.5, 0.7]

for thresh in thresholds:
    y_pred_thresh = (y_pred_proba >= thresh).astype(int)
    approved = y_pred_thresh.sum()
    reject_rate = (1 - y_pred_thresh).sum() / len(y_pred_thresh) * 100
    accuracy = accuracy_score(y_test, y_pred_thresh)
    
    print(f"\n--- THRESHOLD: {thresh:.0%} ---")
    print(f"Applications APPROVED: {approved} ({approved/len(y_pred_thresh)*100:.1f}%)")
    print(f"Applications REJECTED: {len(y_pred_thresh) - approved} ({reject_rate:.1f}%)")
    print(f"Accuracy on test set: {accuracy:.1%}")

## Confusion Matrix: The Errors That Matter

**Accuracy is not enough.** What matters is the *type* of error:

- **False Negatives (FN)**: Good people denied loans. They walk to a competitor bank.
- **False Positives (FP)**: Bad loans approved. The bank loses money and reputation.

In [ ]:
# Confusion matrix analysis
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, 
            xticklabels=['REJECT', 'APPROVE'], yticklabels=['Bad Risk', 'Good Risk'],
            annot_kws={'size': 14, 'fontweight': 'bold'}, ax=ax)
ax.set_ylabel('Actual Credit Risk', fontsize=12, fontweight='bold')
ax.set_xlabel('Model Decision', fontsize=12, fontweight='bold')
ax.set_title('Confusion Matrix: Where Do We Go Wrong?', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nConfusion Matrix Interpretation:")
print(f"  True Negatives (TN):  {tn} — Bad risks correctly rejected")
print(f"  False Positives (FP): {fp} — Bad risks wrongly approved (🚨 LOSS!)")
print(f"  False Negatives (FN): {fn} — Good risks wrongly rejected (😢 OPPORTUNITY LOST)")
print(f"  True Positives (TP):  {tp} — Good risks correctly approved")

## ETHICAL QUESTION: Which Error is Worse?

**Discussion Point:** 

- If you **reject a good applicant**, they go to another bank. You lose market share.
- If you **approve a bad applicant**, the bank loses money. Default rates spike. You face regulators.

**Which matters more? It depends on whose perspective you take:**

From the **bank's perspective**: False positives (bad loans approved) hurt profitability.

From the **borrower's perspective**: False negatives (good people rejected) hurt financial inclusion and opportunity.

**DISCUSS WITH YOUR NEIGHBOR:** If you were running this bank, which error would you prioritize? Why?

---

# ACT 2: NAIVE BAYES — The Independent Thinker

## The Frame

What if each feature **independently voted** on the outcome? 

- "Does the person have a checking account status?" → Votes YES
- "Is employment stable?" → Votes YES
- "Is age suitable?" → Votes YES

Naive Bayes ignores correlations. In the real world, income and savings are correlated — if you have savings, you probably have income. But Naive Bayes treats them as independent. Does this hurt performance?

## Train Naive Bayes

In [ ]:
# Train Naive Bayes classifier
nb_model = GaussianNB()
nb_model.fit(X_train, y_train)
y_pred_nb = nb_model.predict(X_test)

print(f"Naive Bayes trained.")
print(f"\nNaive Bayes Accuracy: {accuracy_score(y_test, y_pred_nb):.1%}")

## Classification Report: Precision & Recall

- **Precision**: Of the loans we approved, what % were actually good?
- **Recall**: Of all good applicants, what % did we catch?

In [ ]:
print("NAIVE BAYES CLASSIFICATION REPORT:")
print("="*50)
print(classification_report(y_test, y_pred_nb, target_names=['Bad Risk', 'Good Risk']))

## POLICY QUESTION: Speed vs. Accuracy

Naive Bayes is **fast and simple**. When you're processing millions of applications, "good enough" might be acceptable.

**Compare:**
- Logistic Regression: More complex, potentially more accurate
- Naive Bayes: Lightning fast, easier to explain, "good enough"

**DISCUSS:** In a high-volume lending operation, is speed worth the potential drop in accuracy? At what point does "good enough" become "not good enough"?

---

# ACT 3: DECISION TREES — The Transparent Judge

## The Frame

What if the algorithm could **explain every decision**? 

In a court of law, a judge can't say "The neural network told me to reject you." But a judge *can* follow a decision tree: "First, we check your checking account status. If it's poor, we look at your employment history..." — and so on.

**Interpretability matters for policy.** It's required by law in some jurisdictions (e.g., GDPR's "right to explanation").

## Train Decision Tree

In [ ]:
# Train decision tree with limited depth for interpretability
dt_model = DecisionTreeClassifier(max_depth=4, random_state=42, min_samples_split=20)
dt_model.fit(X_train, y_train)
y_pred_dt = dt_model.predict(X_test)

print(f"Decision Tree trained with max_depth=4 for readability.")
print(f"Decision Tree Accuracy: {accuracy_score(y_test, y_pred_dt):.1%}")

## Visualize the Decision Tree

This tree can be printed and given to a loan officer. Every decision is traceable.

In [ ]:
fig, ax = plt.subplots(figsize=(20, 10))
plot_tree(dt_model, 
          feature_names=feature_cols,
          class_names=['Bad Risk', 'Good Risk'],
          filled=True,
          rounded=True,
          fontsize=10,
          ax=ax)
plt.title('Decision Tree: The Transparent Judge', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("\nYou can follow every branch. A good credit officer could explain this to any applicant.")

## Feature Importance: What Really Matters?

The tree reveals which features drive decisions.

In [ ]:
# Feature importance
importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': dt_model.feature_importances_
}).sort_values('Importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.viridis(np.linspace(0, 1, len(importance_df)))
ax.barh(importance_df['Feature'], importance_df['Importance'], color=colors, edgecolor='black')
ax.set_xlabel('Importance Score', fontsize=11, fontweight='bold')
ax.set_title('What Does the Tree Use to Decide?', fontsize=12, fontweight='bold')
ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print("\nTop Features in Order:")
for idx, row in importance_df.iterrows():
    print(f"  {row['Feature']}: {row['Importance']:.2%}")

## POLICY INSIGHT: Fairness Through Transparency

**The tree says 'Checking Status' is the most important feature. Does that feel fair?**

Someone who just moved to a new city might not have a local checking account yet. Should their loan be rejected because of *geography*, not character?

This is where **data-driven decisions meet human judgment**. The algorithm is a tool, not a dictator.

---

# THE GRAND FINALE: Model Comparison & Cross-Lab Story

## Compare All Three Models

In [ ]:
# Comprehensive comparison
models_comparison = pd.DataFrame({
    'Model': ['Logistic Regression', 'Naive Bayes', 'Decision Tree'],
    'Accuracy': [
        accuracy_score(y_test, y_pred),
        accuracy_score(y_test, y_pred_nb),
        accuracy_score(y_test, y_pred_dt)
    ],
    'Interpretability': ['Moderate', 'High', 'Very High'],
    'Speed': ['Fast', 'Very Fast', 'Very Fast'],
    'Best For': [
        'Balanced approach',
        'High-volume processing',
        'Regulatory compliance'
    ]
})

print("\nMODEL COMPARISON SUMMARY")
print("="*80)
print(models_comparison.to_string(index=False))
print("\n")

fig, ax = plt.subplots(figsize=(10, 6))
accuracies = models_comparison['Accuracy'].values
colors_models = ['#3498db', '#e74c3c', '#2ecc71']
ax.bar(models_comparison['Model'], accuracies, color=colors_models, edgecolor='black', linewidth=1.5, alpha=0.8)
ax.set_ylabel('Accuracy Score', fontsize=11, fontweight='bold')
ax.set_title('Which Model Performs Best?', fontsize=12, fontweight='bold')
ax.set_ylim([0.7, 0.85])
ax.grid(alpha=0.3, axis='y')

for i, (model, acc) in enumerate(zip(models_comparison['Model'], accuracies)):
    ax.text(i, acc + 0.005, f'{acc:.1%}', ha='center', fontweight='bold')

plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

## WHICH MODEL WOULD YOU DEPLOY? WHY?

**DISCUSS WITH YOUR NEIGHBOR:**

Imagine you're presenting to the bank's board of directors:

- The **CEO** cares about speed and volume.
- The **Chief Risk Officer** cares about minimizing defaults.
- The **Legal Department** cares about being able to explain decisions to regulators.
- The **Marketing Director** cares about approval rates to grow the customer base.

Each would choose a different model. **What would you recommend and why?**

---

# THE CROSS-LAB NARRATIVE: Your Journey Through Data

## Lab 0: Understanding the People

We loaded 1,000 loan applications from Germany in the 1990s. We met:
- A 23-year-old first-time borrower seeking 1,100 DM for a radio/TV
- A 67-year-old retiree with 15,945 DM in savings
- A factory worker, a manager, a student — all with different dreams

**We didn't just see numbers. We saw *people*.** Each row was a human decision to borrow, and a bank's decision to trust (or not).

## Lab 1: Understanding What Drives Credit

We built regression models to predict **credit amounts**. We discovered:
- Duration of employment matters
- Purpose of the loan (car vs. education vs. furniture) matters
- Age and income correlate with credit demand

**We learned the *economics* of borrowing.** Not just patterns in data, but reasons.

## Lab 2: Building Machines That Decide People's Futures

Now we've built classifiers that say **YES or NO** to loan applications.

We've learned:
- Logistic Regression gives us **probabilities** and lets us set thresholds
- Naive Bayes trades assumptions for **speed**
- Decision Trees provide **transparency** that regulators demand

But here's the crucial insight: **None of these models is "the right answer."** They're tools with tradeoffs.

---

# FINAL REFLECTION: What Does This All Mean?

## Core Truths

**1. Every dataset is a collection of human stories.**

The German Credit dataset isn't abstract. It's 1,000 people's attempts to improve their lives through credit. Some succeeded. Some defaulted. All had reasons.

**2. Every model is a set of assumptions about those stories.**

When you choose Logistic Regression over Naive Bayes, you're assuming that feature relationships matter. When you set the approval threshold at 70% instead of 50%, you're assuming some false negatives are worth the reduced false positives.

**3. Every prediction is a policy decision that affects real lives.**

When your model rejects a loan applicant:
- That person doesn't buy the car they needed for their new job
- They can't send their child to the university they dreamed of
- They fall further behind economically

When your model approves a bad loan:
- That person falls into debt they can't escape
- The bank's capital gets tied up
- Trust in the financial system erodes

**There are no neutral decisions.** Every model, every threshold, every feature has winners and losers.

---

# YOUR CHALLENGE: The Policy Brief

Imagine you're presenting to the bank's board of directors. You have 5 minutes. You must make **three recommendations**:

## Recommendation 1: Which Model to Deploy and Why

**Write your answer:**

[Your response here: 2-3 sentences]

## Recommendation 2: What Approval Threshold to Use

Should you approve everyone with >50% probability? >60%? >80%? 

**Consider:**
- What's your target approval rate?
- What's your tolerance for bad loans?
- What's your competitive position (can you afford to be strict)?

**Write your answer:**

[Your response here: 2-3 sentences]

## Recommendation 3: What Safeguards to Put in Place

No model is perfect. What happens when it makes mistakes?

**Consider:**
- Who reviews borderline cases (probability near 50%)?
- How do you handle edge cases the model hasn't seen?
- How do you monitor for bias and fairness?
- What appeals process do rejected applicants have?

**Write your answer:**

[Your response here: 2-3 sentences]

---

# The Final Truth

> **The algorithm is just the beginning. The real work is governance.**

Data science can build the best model in the world, but it takes **business judgment**, **ethics**, **law**, and **human empathy** to deploy it responsibly.

## Your role as a data scientist isn't to optimize accuracy.
## Your role is to help your organization make better decisions — **while being honest about the tradeoffs**.

---

### Credits

**Dataset:** German Credit Dataset (UCI Machine Learning Repository)  
**Theme:** "Tell a story through data" — Cross-lab narrative connecting data exploration → regression → classification  
**Duration:** 45 minutes  
**Audience:** Final-year students and faculty  

**Next Steps:**
- Lab 3: Deep learning and neural networks (coming soon)
- Lab 4: Time series and forecasting (coming soon)